### Testing Big Query

THis is a work in progress. Intended to show how to call functions

In [27]:
import os
from dotenv import load_dotenv
import json
import pandas as pd
import numpy as np
from google.cloud import bigquery
import pydata_google_auth

from scripts.queries import *
from scripts.constants import SITE_MAPPINGS, SITE_BURN_DATATYPES, SITE_GENERATION_DATATYPES
from scripts.data_pull_functions import run_date_parameterized_query, run_generation_query

load_dotenv()

True

In [28]:
# this will open a browser to confirm access when running for the first time
credentials = pydata_google_auth.get_user_credentials(
    scopes=["https://www.googleapis.com/auth/cloud-platform"],
    auth_local_webserver=True,
)

In [29]:
client = bigquery.Client(project="bepc-prj-energy-prod", credentials=credentials )

## Gas Burn Data

In [4]:
#gas_daily_df = run_date_parameterized_query(client, GAS_BURN_QUERY, '2024-07-23', '2026-07-22', SITE_BURN_DATATYPES)
gas_daily_df = run_date_parameterized_query(client, GAS_BURN_QUERY, '2024-01-01', '2026-07-31', SITE_BURN_DATATYPES)

In [5]:
gas_daily_df['site'] = gas_daily_df['marketarea'].replace(SITE_MAPPINGS)

gas_daily_df = gas_daily_df[['gas_day', 'site', 'energy']]
gas_daily_df.head()

,gas_day,site,energy
0,2024-01-01,DCS,26760.0
1,2024-01-01,GGS,3592.0
2,2024-01-01,LCS,19535.0
3,2024-01-01,PGS,20515.0
4,2024-01-01,CGS,3575.0


In [6]:
# Validity check against 'checkpoint sql 1' file
gas_daily_df.loc[:, ['site', 'energy']].groupby(by=['site']).agg('sum') # matches

,energy
site,
CGS,6280400.0
DCS,23590674.0
GGS,5186154.0
LCS,24005720.0
PGS,45621135.0


## Generation Data

Goal is to write a single function that
1) runs pgs and non-pgs queries using the run_energy_query
2) concatenate the queries
3) make any adjustments to both table at the same time

In [ ]:
## This has been converted to a function in the data_pull_functions.py file
def run_generation_query(client: bigquery.Client, query_strings: list, start: str, end: str, col_dtypes: dict = None):
    """
    Function designed to pull generation data based on multiple SQL queries. SQL queries must contain the same column headers

    Parameters:
        client: Session bigquery.client object
        query_strings: SQL queries
        start: Earliest date of data to pull from database
        end: Latest date of data to pull from database
        col_type: dictionary containing column name to datatype mappings

    """

    dfs = {}
    for idx, query in enumerate(query_strings):
    
        df_name = f'df{idx}'
        dfs[df_name] = run_date_parameterized_query(client, query, start, end)
    
    dfs_appended = pd.concat(dfs, ignore_index=True)

    if col_dtypes: 
        dfs_appended = dfs_appended.astype(col_dtypes)

    return dfs_appended



In [21]:
test = run_generation_query(client, [PGS_GENERATION_QUERY, NON_PGS_GENERATION_QUERY], '2025-01-01', '2025-12-31', SITE_GENERATION_DATATYPES)

#test = test.astype(SITE_GENERATION_DATATYPES)
print(test.shape)
#test.head(-5)
#test.to_csv(r'./output-data/non_pgs_only check.csv', index=False)

(12960, 26)


In [10]:
test[(test['loadshape']=='WAUE.BEPM.PGS1 - Net Generation-5m')]

,begtime,loadshape,he01,he02,he03,he04,he05,he06,he07,he08,...,he15,he16,he17,he18,he19,he20,he21,he22,he23,he24
0,2025-01-01,WAUE.BEPM.PGS1 - Net Generation-5m,38.4,38.4,38.4,38.4,38.4,38.4,38.4,38.4,...,38.4,38.4,38.4,38.4,38.4,38.4,38.4,38.4,38.4,38.4
1,2025-01-02,WAUE.BEPM.PGS1 - Net Generation-5m,38.4,38.4,38.4,38.4,38.4,38.4,38.4,38.4,...,38.4,38.4,38.4,38.4,38.4,38.4,38.4,38.4,38.4,38.4
2,2025-01-03,WAUE.BEPM.PGS1 - Net Generation-5m,38.4,38.4,38.4,38.4,38.4,38.4,38.4,38.4,...,38.4,38.4,38.4,38.4,38.4,38.4,38.4,38.4,38.4,38.4
3,2025-01-04,WAUE.BEPM.PGS1 - Net Generation-5m,38.4,38.4,38.4,38.4,38.4,38.4,38.4,38.4,...,38.4,38.4,38.4,38.4,38.4,38.4,38.4,38.4,38.4,38.4
4,2025-01-05,WAUE.BEPM.PGS1 - Net Generation-5m,38.4,38.4,38.4,38.4,38.4,38.4,38.4,38.4,...,38.4,38.4,38.4,38.4,38.4,38.4,38.4,38.4,38.4,38.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
360,2025-12-27,WAUE.BEPM.PGS1 - Net Generation-5m,29.1,3.9,0.0,0.0,0.0,0.0,7.0,29.2,...,34.2,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
361,2025-12-28,WAUE.BEPM.PGS1 - Net Generation-5m,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,24.7,44.1,33.4,36.9,39.9,33.6,33.7,33.8
362,2025-12-29,WAUE.BEPM.PGS1 - Net Generation-5m,30.2,32.3,33.7,32.2,40.7,29.5,30.6,29.4,...,42.6,40.9,42.4,42.8,44.2,44.2,43.8,43.7,43.7,43.3
363,2025-12-30,WAUE.BEPM.PGS1 - Net Generation-5m,39.8,29.3,29.3,29.2,29.3,29.3,29.8,32.3,...,29.3,29.4,30.9,29.2,29.3,30.5,29.8,29.7,29.3,30.6


In [12]:
#test['loadshape'].unique()
conditions = [
    test["loadshape"].str.upper().str.contains("DCS"),
    test["loadshape"].str.upper().str.contains("LCS"),
    test["loadshape"].str.upper().str.contains("PGS"),
    test["loadshape"].str.upper().str.contains("GGS"),
    test["loadshape"].str.upper().str.contains("CULBERTSON")
]

choices = ["DCS", "LCS", "PGS", "GGS", "CGS"]

test['site'] = np.select(conditions, choices, default="N/A")

print(test.shape)


(12960, 27)


In [37]:
test.head()

,begtime,loadshape,he01,he02,he03,he04,he05,he06,he07,he08,...,he16,he17,he18,he19,he20,he21,he22,he23,he24,site
0,2025-02-05,WAUE.BEPM.CULBERTSON1 - Net Generation,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,63.5,63.3,64.3,59.9,59.3,59.6,59.7,62.7,60.1,CGS
1,2025-02-11,WAUE.BEPM.CULBERTSON1 - Net Generation,61.1,60.6,59.3,63.2,71.6,72.3,72.4,66.5,...,82.8,85.5,92.8,94.2,94.3,94.4,94.3,78.2,94.5,CGS
2,2025-02-17,WAUE.BEPM.CULBERTSON1 - Net Generation,28.5,28.6,28.6,28.6,28.6,28.7,28.2,28.1,...,59.9,60.1,84.4,78.1,77.6,61.7,60.2,79.8,62.6,CGS
3,2025-02-20,WAUE.BEPM.CULBERTSON1 - Net Generation,60.4,63.6,60.5,60.0,60.0,65.2,67.9,48.3,...,59.2,60.1,60.1,63.1,59.8,59.8,59.2,59.8,60.1,CGS
4,2025-02-09,WAUE.BEPM.DCS1 - Net Generation,182.0,214.0,216.0,190.0,190.0,208.0,199.0,203.0,...,190.0,190.0,190.0,196.0,194.0,190.0,190.0,190.0,190.0,DCS


In [13]:
#Unpivoting columns using melt() function
hour_cols = [column for column in test.columns if column.startswith("he")]
test_hourly_df = test.melt(id_vars=["begtime", "site", "loadshape"], value_vars=hour_cols, var_name="hour", value_name="hourly_mw")

print(test_hourly_df.shape)
test_hourly_df.head()

(311040, 5)


,begtime,site,loadshape,hour,hourly_mw
0,2025-01-01,PGS,WAUE.BEPM.PGS1 - Net Generation-5m,he01,38.4
1,2025-01-02,PGS,WAUE.BEPM.PGS1 - Net Generation-5m,he01,38.4
2,2025-01-03,PGS,WAUE.BEPM.PGS1 - Net Generation-5m,he01,38.4
3,2025-01-04,PGS,WAUE.BEPM.PGS1 - Net Generation-5m,he01,38.4
4,2025-01-05,PGS,WAUE.BEPM.PGS1 - Net Generation-5m,he01,38.4


In [14]:
test_hourly_df['hourly_mw'].sum()

np.float64(4160239.31)

In [15]:
#Create hour (numeric column) for Datetime creation
test_hourly_df["hour_num"] = test_hourly_df["hour"].str[-2:].astype(int)

test_hourly_df = test_hourly_df.sort_values(by = ['site', 'loadshape', 'begtime', 'hour'], ascending=[False, False, True, True])
test_hourly_df.head()


,begtime,site,loadshape,hour,hourly_mw,hour_num
9035,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he01,0.0,1
21995,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he02,0.0,2
34955,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he03,0.0,3
47915,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he04,0.0,4
60875,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he05,0.0,5


In [16]:
test_hourly_df["datetime"] = (pd.to_datetime(test_hourly_df["begtime"]) + pd.to_timedelta(test_hourly_df["hour_num"] - 0, unit="h"))
test_hourly_df.head(24)

,begtime,site,loadshape,hour,hourly_mw,hour_num,datetime
9035,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he01,0.0,1,2025-04-01 01:00:00
21995,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he02,0.0,2,2025-04-01 02:00:00
34955,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he03,0.0,3,2025-04-01 03:00:00
47915,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he04,0.0,4,2025-04-01 04:00:00
60875,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he05,0.0,5,2025-04-01 05:00:00
73835,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he06,0.0,6,2025-04-01 06:00:00
86795,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he07,0.0,7,2025-04-01 07:00:00
99755,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he08,0.0,8,2025-04-01 08:00:00
112715,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he09,0.0,9,2025-04-01 09:00:00
125675,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he10,0.0,10,2025-04-01 10:00:00


In [20]:
test_hourly_df['gas_day'] = (pd.to_datetime(test_hourly_df["datetime"]) - pd.Timedelta(hours=10)).dt.date

test_hourly_df[(test_hourly_df['loadshape']=='WAUE.BEPM.PGS1 - Net Generation-5m') & (test_hourly_df['begtime']=='2025-07-01')].head(48)
#test_hourly_df[(test_hourly_df['loadshape']=='WAUE.BEPM.DCS - Net Generation') & (test_hourly_df['begtime']=='2026-07-27')].head(48)


,begtime,site,loadshape,hour,hourly_mw,hour_num,datetime,gas_day
181,2025-07-01,PGS,WAUE.BEPM.PGS1 - Net Generation-5m,he01,0.0,1,2025-07-01 01:00:00,2025-06-30
13141,2025-07-01,PGS,WAUE.BEPM.PGS1 - Net Generation-5m,he02,0.0,2,2025-07-01 02:00:00,2025-06-30
26101,2025-07-01,PGS,WAUE.BEPM.PGS1 - Net Generation-5m,he03,0.0,3,2025-07-01 03:00:00,2025-06-30
39061,2025-07-01,PGS,WAUE.BEPM.PGS1 - Net Generation-5m,he04,0.0,4,2025-07-01 04:00:00,2025-06-30
52021,2025-07-01,PGS,WAUE.BEPM.PGS1 - Net Generation-5m,he05,0.0,5,2025-07-01 05:00:00,2025-06-30
64981,2025-07-01,PGS,WAUE.BEPM.PGS1 - Net Generation-5m,he06,0.0,6,2025-07-01 06:00:00,2025-06-30
77941,2025-07-01,PGS,WAUE.BEPM.PGS1 - Net Generation-5m,he07,0.0,7,2025-07-01 07:00:00,2025-06-30
90901,2025-07-01,PGS,WAUE.BEPM.PGS1 - Net Generation-5m,he08,0.0,8,2025-07-01 08:00:00,2025-06-30
103861,2025-07-01,PGS,WAUE.BEPM.PGS1 - Net Generation-5m,he09,0.0,9,2025-07-01 09:00:00,2025-06-30
116821,2025-07-01,PGS,WAUE.BEPM.PGS1 - Net Generation-5m,he10,0.0,10,2025-07-01 10:00:00,2025-07-01


In [26]:
test_hourly_df['gas_day'] = (pd.to_datetime(test_hourly_df["datetime"]) - pd.Timedelta(hours=10)).dt.date

test_hourly_df[(test_hourly_df['loadshape']=='WAUE.BEPM.PGS1 - Net Generation-5m') & (test_hourly_df['begtime']=='2026-07-01')].head(48)

,begtime,site,loadshape,hour,hourly_mw,hour_num,datetime,gas_day
0,2026-07-01,PGS,WAUE.BEPM.PGS1 - Net Generation-5m,he01,29.5,1,2026-07-01 01:00:00,2026-06-30
1116,2026-07-01,PGS,WAUE.BEPM.PGS1 - Net Generation-5m,he02,29.5,2,2026-07-01 02:00:00,2026-06-30
2232,2026-07-01,PGS,WAUE.BEPM.PGS1 - Net Generation-5m,he03,3.5,3,2026-07-01 03:00:00,2026-06-30
3348,2026-07-01,PGS,WAUE.BEPM.PGS1 - Net Generation-5m,he04,0.0,4,2026-07-01 04:00:00,2026-06-30
4464,2026-07-01,PGS,WAUE.BEPM.PGS1 - Net Generation-5m,he05,0.0,5,2026-07-01 05:00:00,2026-06-30
5580,2026-07-01,PGS,WAUE.BEPM.PGS1 - Net Generation-5m,he06,0.0,6,2026-07-01 06:00:00,2026-06-30
6696,2026-07-01,PGS,WAUE.BEPM.PGS1 - Net Generation-5m,he07,9.1,7,2026-07-01 07:00:00,2026-06-30
7812,2026-07-01,PGS,WAUE.BEPM.PGS1 - Net Generation-5m,he08,29.6,8,2026-07-01 08:00:00,2026-06-30
8928,2026-07-01,PGS,WAUE.BEPM.PGS1 - Net Generation-5m,he09,29.8,9,2026-07-01 09:00:00,2026-06-30
10044,2026-07-01,PGS,WAUE.BEPM.PGS1 - Net Generation-5m,he10,29.9,10,2026-07-01 10:00:00,2026-07-01


In [41]:
test_hourly_df['hourly_mw'].sum()

np.float64(2943558.563)

In [24]:
def clean_generation_unit_data(df: pd.DataFrame):
    """
    Function that takes the unit generation data as input, creates a site variable, calculates gas day, and aggregates by site

    Parameters:
        df: dataframe containing hourly unit level generation data
    """
    unit_df = df.copy()

    #adding a site column based on the loadshape name
    conditions = [
        unit_df["loadshape"].str.upper().str.contains("DCS"),
        unit_df["loadshape"].str.upper().str.contains("LCS"),
        unit_df["loadshape"].str.upper().str.contains("PGS"),
        unit_df["loadshape"].str.upper().str.contains("GGS"),
        unit_df["loadshape"].str.upper().str.contains("CGS")
    ]

    choices = ["DCS", "LCS", "PGS", "GGS", "CGS"]

    unit_df['site'] = np.select(conditions, choices, default="N/A")

    #Unpivoting columns using melt() function
    hour_cols = [column for column in unit_df.columns if column.startswith("he")]
    hourly_unit_generation_df = unit_df.melt(id_vars=["begtime", "site", "loadshape"], value_vars=hour_cols, var_name="hour", value_name="hourly_mw")
    #print(hourly_unit_generation_df['hourly_mw'].sum()) # may take out

    #Create hour (numeric column) for Datetime creation
    hourly_unit_generation_df["hour_num"] = hourly_unit_generation_df["hour"].str[-2:].astype(int)

    #create a datetime column and related gas_day column
    hourly_unit_generation_df["datetime"] = (pd.to_datetime(hourly_unit_generation_df["begtime"]) + pd.to_timedelta(hourly_unit_generation_df["hour_num"] - 0, unit="h"))
    hourly_unit_generation_df['gas_day'] = (pd.to_datetime(hourly_unit_generation_df["datetime"]) - pd.Timedelta(hours=10)).dt.date
    hourly_unit_generation_df['gas_day'] = pd.to_datetime(hourly_unit_generation_df['gas_day'])
    #print(hourly_unit_generation_df['hourly_mw'].sum()) # may take out


    #Reordering columns for visual 
    hourly_unit_generation_df = (
        hourly_unit_generation_df[["datetime","gas_day", "hour", "site", "loadshape", "hourly_mw"]]
        .sort_values(by = ['site', 'datetime'], ascending=[False, True])
    )
    print(hourly_unit_generation_df['hourly_mw'].sum()) # may take out


    # Aggregating by site
    hourly_site_generation_df = (
        hourly_unit_generation_df.groupby(["datetime", "gas_day", "hour", "site"], as_index=False)["hourly_mw"]
        .sum()
        .rename(columns={"hourly_mw": "hourly_site_gen_mw"})
        .sort_values(by = ['site', 'datetime'], ascending=[False, True])
    )
    #print(hourly_site_generation_df['hourly_site_gen_mw'].sum()) # may take out


    daily_site_generation_by_gas_day_df = (
        hourly_unit_generation_df.groupby(["gas_day", "site"], as_index=False)["hourly_mw"]
        .sum()
        .rename(columns={"hourly_mw": "hourly_site_gen_mw"})
        .sort_values(by = ['site', 'gas_day'], ascending=[False, True])
    )
    #print(daily_site_generation_by_gas_day_df['hourly_site_gen_mw'].sum()) # may take out


    daily_site_generation_df = (
        hourly_unit_generation_df.groupby(["datetime", "site"], as_index=False)["hourly_mw"]
        .sum()
        .rename(columns={"hourly_mw": "hourly_site_gen_mw"})
        .sort_values(by = ['site', 'datetime'], ascending=[False, True])
    )
    #print(daily_site_generation_df['hourly_site_gen_mw'].sum()) # may take out



    return {
        'hourly_unit_generation_df': hourly_unit_generation_df, 
        'hourly_site_generation_df': hourly_site_generation_df, 
        'daily_site_generation_by_gas_day_df': daily_site_generation_by_gas_day_df, 
        'daily_site_generation_df': daily_site_generation_df
    }


In [26]:
test_cleaned = clean_generation_unit_data(test)
test_cleaned['hourly_site_generation_df']


4160239.31


,datetime,gas_day,hour,site,hourly_site_gen_mw
4,2025-01-01 01:00:00,2024-12-31,he01,PGS,100.5
9,2025-01-01 02:00:00,2024-12-31,he02,PGS,85.4
14,2025-01-01 03:00:00,2024-12-31,he03,PGS,121.2
19,2025-01-01 04:00:00,2024-12-31,he04,PGS,121.1
24,2025-01-01 05:00:00,2024-12-31,he05,PGS,124.8
...,...,...,...,...,...
43775,2025-12-31 20:00:00,2025-12-31,he20,DCS,190.0
43780,2025-12-31 21:00:00,2025-12-31,he21,DCS,194.0
43785,2025-12-31 22:00:00,2025-12-31,he22,DCS,243.0
43790,2025-12-31 23:00:00,2025-12-31,he23,DCS,296.0


In [81]:
non_pgs_gen_only = test_cleaned['hourly_site_generation_df']
print(non_pgs_gen_only['hourly_site_gen_mw'].sum())
non_pgs_gen_only.head()


4160239.31


,datetime,gas_day,hour,site,hourly_site_gen_mw
4,2025-01-01 01:00:00,2024-12-31,he01,PGS,100.5
9,2025-01-01 02:00:00,2024-12-31,he02,PGS,85.4
14,2025-01-01 03:00:00,2024-12-31,he03,PGS,121.2
19,2025-01-01 04:00:00,2024-12-31,he04,PGS,121.1
24,2025-01-01 05:00:00,2024-12-31,he05,PGS,124.8


In [82]:
non_pgs_gen_only['datetime'].max()

Timestamp('2026-01-01 00:00:00')

In [83]:
non_pgs_gen_only[(non_pgs_gen_only['site']=='PGS') & (non_pgs_gen_only['datetime']>='2025-01-01') & (non_pgs_gen_only['datetime']<='2026-01-01')]['hourly_site_gen_mw'].sum()

np.float64(1216680.747)

In [16]:
nonpgs_generation_df = run_date_parameterized_query(client, PGS_GENERATION_QUERY, '2024-07-23', '2026-07-22')
print(nonpgs_generation_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 17426 entries, 0 to 17425
Data columns (total 26 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   begtime    17426 non-null  datetime64[us]
 1   loadshape  17426 non-null  str           
 2   he1        17426 non-null  object        
 3   he2        17426 non-null  object        
 4   he3        17384 non-null  object        
 5   he4        17426 non-null  object        
 6   he5        17426 non-null  object        
 7   he6        17426 non-null  object        
 8   he7        17426 non-null  object        
 9   he8        17426 non-null  object        
 10  he9        17426 non-null  object        
 11  he10       17426 non-null  object        
 12  he11       17426 non-null  object        
 13  he12       17426 non-null  object        
 14  he13       17426 non-null  object        
 15  he14       17426 non-null  object        
 16  he15       17426 non-null  object        
 17  he16

In [17]:
pgs_generation_df = run_date_parameterized_query(client, NON_PGS_GENERATION_QUERY, '2024-07-23', '2026-07-22')
print(pgs_generation_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 7300 entries, 0 to 7299
Data columns (total 26 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   begtime    7300 non-null   datetime64[us]
 1   loadshape  7300 non-null   str           
 2   he1        7300 non-null   object        
 3   he2        7300 non-null   object        
 4   he3        7280 non-null   object        
 5   he4        7300 non-null   object        
 6   he5        7300 non-null   object        
 7   he6        7300 non-null   object        
 8   he7        7300 non-null   object        
 9   he8        7300 non-null   object        
 10  he9        7290 non-null   object        
 11  he10       7290 non-null   object        
 12  he11       7290 non-null   object        
 13  he12       7290 non-null   object        
 14  he13       7290 non-null   object        
 15  he14       7290 non-null   object        
 16  he15       7290 non-null   object        
 17  he16  

## Non-PGS Data Generation Data

In [18]:
query_job = client.query(NON_PGS_GENERATION_QUERY)
non_pgs_generation_df = query_job.to_dataframe(create_bqstorage_client=False)
non_pgs_generation_df.head()

BadRequest: 400 Query parameter 'start_date' not found at [31:29]; reason: invalidQuery, location: query, message: Query parameter 'start_date' not found at [31:29]

Location: us-central1
Job ID: 27359546-26e2-4b7a-9145-269d834d0f16


In [19]:
## updating column types from the get go
hour_end_columns = [col for col in pgs_generation_df.columns if col.startswith('he')]

pgs_datatypes = {col: 'float64' for col in hour_end_columns}
pgs_datatypes.update({
    'begtime': 'datetime64[ns]', 
    'loadshape': 'str'
})


In [20]:
pgs_datatypes

{'he1': 'float64',
 'he2': 'float64',
 'he3': 'float64',
 'he4': 'float64',
 'he5': 'float64',
 'he6': 'float64',
 'he7': 'float64',
 'he8': 'float64',
 'he9': 'float64',
 'he10': 'float64',
 'he11': 'float64',
 'he12': 'float64',
 'he13': 'float64',
 'he14': 'float64',
 'he15': 'float64',
 'he16': 'float64',
 'he17': 'float64',
 'he18': 'float64',
 'he19': 'float64',
 'he20': 'float64',
 'he21': 'float64',
 'he22': 'float64',
 'he23': 'float64',
 'he24': 'float64',
 'begtime': 'datetime64[ns]',
 'loadshape': 'str'}

In [ ]:

pgs_generation_df = non_pgs_generation_df.astype(pgs_datatypes)

In [54]:
non_pgs_generation_df['he1'].sum()

np.float64(155525.637)

## PGS Generation Data

In [6]:
query_job = client.query(PGS_GENERATION_QUERY)
pgs_generation_df = query_job.to_dataframe()
pgs_generation_df.head()

c:\Users\A105158\OneDrive - Basin Electric Power Cooperative\Desktop\Python Projects\gas-burn-by-site\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,begtime,loadshape,he1,he2,he3,he4,he5,he6,he7,he8,...,he15,he16,he17,he18,he19,he20,he21,he22,he23,he24
0,2024-07-24,WAUE.BEPM.PGS1 - Net Generation-5m,29.600000000,2.500000000,0E-9,0E-9,0E-9,3.500000000,29.700000000,29.600000000,...,34.000000000,33.900000000,0.300000000,0E-9,0E-9,0E-9,0E-9,0E-9,0E-9,0E-9
1,2024-07-25,WAUE.BEPM.PGS1 - Net Generation-5m,0E-9,0E-9,0E-9,0E-9,0E-9,2.100000000,25.800000000,4.900000000,...,0E-9,0E-9,0E-9,0E-9,0E-9,0E-9,5.500000000,36.900000000,31.700000000,29.500000000
2,2024-07-26,WAUE.BEPM.PGS1 - Net Generation-5m,29.700000000,33.900000000,30.400000000,29.600000000,35.200000000,31.400000000,30.000000000,30.500000000,...,29.400000000,29.700000000,29.800000000,29.700000000,29.900000000,29.900000000,29.900000000,29.500000000,32.800000000,32.300000000
3,2024-07-27,WAUE.BEPM.PGS1 - Net Generation-5m,22.600000000,0E-9,0E-9,0E-9,3.400000000,32.200000000,29.500000000,34.100000000,...,32.800000000,33.300000000,33.500000000,32.300000000,31.900000000,30.400000000,29.700000000,30.600000000,29.700000000,29.300000000
4,2024-07-28,WAUE.BEPM.PGS1 - Net Generation-5m,30.700000000,1.100000000,0E-9,0E-9,0E-9,0E-9,0E-9,0.700000000,...,39.700000000,39.500000000,39.200000000,36.600000000,39.400000000,39.400000000,34.100000000,36.700000000,36.400000000,37.200000000


In [58]:
hour_end_columns = [col for col in pgs_generation_df.columns if col.startswith('he')]

pgs_datatypes = {col: 'float64' for col in hour_end_columns}
pgs_datatypes.update({
    'begtime': 'datetime64[ns]', 
    'loadshape': 'str'
})

pgs_generation_df = pgs_generation_df.astype(pgs_datatypes)

In [59]:
pgs_generation_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12139 entries, 0 to 12138
Data columns (total 26 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   begtime    12139 non-null  datetime64[ns]
 1   loadshape  12139 non-null  str           
 2   he1        12139 non-null  float64       
 3   he2        12139 non-null  float64       
 4   he3        12119 non-null  float64       
 5   he4        12139 non-null  float64       
 6   he5        12139 non-null  float64       
 7   he6        12139 non-null  float64       
 8   he7        12139 non-null  float64       
 9   he8        12139 non-null  float64       
 10  he9        12139 non-null  float64       
 11  he10       12139 non-null  float64       
 12  he11       12139 non-null  float64       
 13  he12       12139 non-null  float64       
 14  he13       12139 non-null  float64       
 15  he14       12139 non-null  float64       
 16  he15       12139 non-null  float64       
 17  he16

In [60]:
pgs_generation_df['he1'].sum()

np.float64(66189.975)